# Swin Transformer v2 画像・タグ学習システム (Google Colab版)

Google Drive 内の **Cryptomator 暗号化保管庫 (Vault)** から **WebDAV** を介してインメモリでストリーミング学習を行います。

### 特徴
- **SSD/ディスク非保存 (Zero-Disk Persistence)**: Colab のローカルディスクに平文画像を一切保存せず、Cryptomator CLI の内蔵 WebDAV サーバーから直接 RAM 上にストリーミングして学習します。
- **Google Drive 連携**: Google Drive 上にある暗号化 Vault ファイルを直接参照します。
- **GPU 高速化**: Colab の GPU (T4 / V100 / A100 / L4) をフル活用し、高速な推論・学習を実行します。

## 1. Google Drive のマウント & GPU 確認

In [ ]:
# Google Drive をマウント
from google.colab import drive
drive.mount('/content/drive')

# GPU の確認
!nvidia-smi

## 2. 環境構築 & 依存パッケージのインストール

GitHub リポジトリからコードを取得するか、Google Drive 上のコードを使用します。

In [ ]:
import os

# 作業ディレクトリの設定
# (Google Drive 内にプロジェクトフォルダがある場合はそのパスに変更してください)
WORKDIR = '/content/image2tag'

if not os.path.exists(WORKDIR):
    # リポジトリをクローンする場合 (必要に応じてご自身のリポジトリ URL に変更してください)
    # !git clone https://github.com/your-username/image2tag.git {WORKDIR}
    os.makedirs(WORKDIR, exist_ok=True)

%cd {WORKDIR}

# 必要な Python パッケージのインストール
!pip install -q timm rule34Py python-dotenv pyyaml requests pillow tqdm

## 3. Cryptomator CLI のセットアップ

Linux 向け最新の Cryptomator CLI をダウンロードし、セットアップします。

In [ ]:
%%bash
# JRE (Java 実行環境) および unzip のインストール
apt-get update -qq && apt-get install -y -qq default-jre unzip > /dev/null

# Cryptomator CLI のダウンロードとインストール (v0.6.2)
CLI_URL="https://github.com/cryptomator/cli/releases/download/0.6.2/cryptomator-cli-0.6.2-linux-x64.zip"
if [ ! -f "/usr/local/bin/cryptomator-cli" ]; then
    echo "Cryptomator CLI をダウンロード中..."
    wget -q -O /tmp/cryptomator-cli.zip $CLI_URL
    mkdir -p /opt/cryptomator-cli
    unzip -q -o /tmp/cryptomator-cli.zip -d /opt/cryptomator-cli
    chmod +x /opt/cryptomator-cli/bin/cryptomator-cli
    ln -sf /opt/cryptomator-cli/bin/cryptomator-cli /usr/local/bin/cryptomator-cli
    rm /tmp/cryptomator-cli.zip
    echo "インストール完了！"
fi

cryptomator-cli --version

## 4. Google Drive 上の Vault をアンロック & WebDAV 起動

Cryptomator CLI の WebDAV モードを起動し、Python スクリプトから自動接続できるように設定します。

In [ ]:
import getpass
import os
import re
import subprocess
import time
import yaml

# ============================================================
# Google Drive 内の Vault (保管庫) フォルダパスを指定してください
# 例: '/content/drive/MyDrive/MyVault' など
# ============================================================
VAULT_PATH = '/content/drive/MyDrive/MyVault'  # ← あなたの Vault パスに変更してください

if not os.path.exists(VAULT_PATH):
    print(f"警告: Vault パスが見つかりません: {VAULT_PATH}")
    print("Google Drive 内の正確なパスを確認してください。")
else:
    # パスワードを安全に入力 (画面に平文表示されません)
    vault_password = getpass.getpass("Cryptomator Vault のパスワードを入力してください: ")

    # 環境変数にセットして cryptomator-cli に渡す
    env = os.environ.copy()
    env["VAULT_PW"] = vault_password

    print("Cryptomator WebDAV サーバーを起動中...")
    cmd = [
        "cryptomator-cli",
        "unlock",
        "--mounter", "org.cryptomator.frontend.webdav.mount.FallbackMounter",
        "--password:env=VAULT_PW",
        VAULT_PATH
    ]

    # バックグラウンドプロセスとして起動
    crypto_proc = subprocess.Popen(
        cmd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )

    # 出力から WebDAV URL を自動検出
    webdav_url = None
    start_time = time.time()
    while time.time() - start_time < 30:
        line = crypto_proc.stdout.readline()
        if line:
            print(line.strip())
            match = re.search(r'(http://(?:localhost|127\.0\.1):\d+/[^\s]+)', line)
            if match:
                webdav_url = match.group(1).rstrip("/")
                break
        if crypto_proc.poll() is not None:
            break
        time.sleep(0.1)

    if webdav_url:
        print(f"\n[成功] Cryptomator WebDAV サーバー起動完了: {webdav_url}")
        # config.yaml を自動更新
        config_path = "config.yaml"
        if not os.path.exists(config_path) and os.path.exists("example.config.yaml"):
            with open("example.config.yaml", "r", encoding="utf-8") as f:
                cfg = yaml.safe_load(f)
        else:
            with open(config_path, "r", encoding="utf-8") as f:
                cfg = yaml.safe_load(f) or {}

        cfg.setdefault("storage", {})["webdav_url"] = webdav_url
        with open(config_path, "w", encoding="utf-8") as f:
            yaml.dump(cfg, f, default_flow_style=False, allow_unicode=True)
        print(f"config.yaml の webdav_url を自動設定しました。")
    else:
        print("\n[エラー] WebDAV URL の取得に失敗しました。パスワードまたは Vault パスを確認してください。")

## 5. 保存済みデータセットの一覧確認

Cryptomator 内に暗号化保存されているデータセット一覧を表示します。

In [ ]:
!python main.py list

## 6. (任意) 新しいデータセットの収集

Rule34 から画像を収集し、Google Drive 上の Cryptomator Vault に直接暗号化保存します。

In [ ]:
# Rule34 API キーを環境変数または .env に設定している場合のみ実行してください
# 例: 'overwatch -ai_generated' で 500枚収集
# !python main.py download --dataset-name "test_overwatch" --tags "overwatch -ai_generated" --limit 500

## 7. Swin Transformer v2 学習の実行 (GPU 高速化)

Cryptomator の WebDAV から画像を直接 RAM 上に並列プリロードし、RTX/T4/V100/A100 GPU で学習を実行します。
Colab のローカルディスクには平文画像は一切書き込まれません。

In [ ]:
# 省メモリモード (low_vram) または 高精度モード (high_quality) を指定可能
!python main.py train \
    --dataset-name "test_overwatch" \
    --preset high_quality \
    --min-frequency 5 \
    --epochs 20 \
    --batch-size 32

## 8. 学習済みモデルによる推論（タグ予測）

学習したチェックポイント（`.pth`）を使って、画像ファイルまたは Web 上の画像 URL からタグを予測します。

In [ ]:
# 例: Web 上の画像またはアップロード画像に対して推論
TEST_IMAGE_URL = "https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg"  # テスト用画像URL

!python main.py predict \
    --checkpoint "./checkpoints/test_overwatch_best_model.pth" \
    --image "{TEST_IMAGE_URL}" \
    --threshold 0.35 \
    --top-k 15

## 9. クリーンアップ (Cryptomator の安全なロック・終了)

学習作業終了後、バックグラウンドの WebDAV サーバーを停止し、Vault を安全にロックします。

In [ ]:
try:
    if 'crypto_proc' in globals() and crypto_proc.poll() is None:
        crypto_proc.terminate()
        crypto_proc.wait(timeout=5)
        print("Cryptomator WebDAV サーバーを正常に停止し、Vault をロックしました。")
    else:
        print("Cryptomator プロセスは実行されていません。")
except Exception as e:
    print(f"停止時エラー: {e}")